# URBAN FLOOD STRESS | 311 Raw Processing

Lee los `raw` de 311 desde `/home/map10194/Documents/ml4c/ml4c-pro/data/raw`, los une como un solo dataset y arranca el procesamiento.

Fuentes:
- `311_2010_2019`
- `311_2020_present`
- `311_2020_present_parts`

Requisito:
- `pyarrow`


Notas breves:
- No descarga nada.
- El notebook usa el esquema raw local y empieza con una agregación mensual por borough y complaint type.


In [3]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

try:
    import pyarrow.dataset as ds
except ModuleNotFoundError as exc:
    raise RuntimeError(
        'This notebook requires pyarrow. Install the environment from environment.yml.'
    ) from exc


def find_project_root() -> Path:
    """Locate the repository root from the current notebook."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'data').exists():
            return candidate
    return current


ROOT = find_project_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from project_name.requests_311 import NYC_311_COLUMNS, clean_311_frame

RAW_311_ROOT = Path('/home/map10194/Documents/ml4c/ml4c-pro/data/raw')
OUT_ROOT = ROOT / 'data' / 'temporal' / '311'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

print(ROOT)
print(RAW_311_ROOT)
print(OUT_ROOT)


/home/map10194/Documents/urban-flood-stress
/home/map10194/Documents/ml4c/ml4c-pro/data/raw
/home/map10194/Documents/urban-flood-stress/data/temporal/311


In [4]:
def discover_311_raw_files(raw_root: Path = RAW_311_ROOT) -> list[Path]:
    """Collect raw 311 parquet files from the external raw data tree."""
    folders = [
        raw_root / '311_2010_2019',
        raw_root / '311_2020_present',
        raw_root / '311_2020_present_parts',
    ]
    paths: list[Path] = []
    for folder in folders:
        if folder.exists():
            paths.extend(sorted(folder.glob('*.parquet')))
    return paths


raw_paths = discover_311_raw_files()
if not raw_paths:
    raise FileNotFoundError('No raw 311 parquet files were found.')

raw_311 = ds.dataset([str(path) for path in raw_paths], format='parquet')
available_columns = [column for column in NYC_311_COLUMNS if column in raw_311.schema.names]
missing_columns = [column for column in NYC_311_COLUMNS if column not in raw_311.schema.names]

print(f'raw files: {len(raw_paths)}')
print(f'rows (metadata count): {raw_311.count_rows()}')
print(f'available columns: {available_columns}')
if missing_columns:
    print(f'missing columns: {missing_columns}')
else:
    print('missing columns: none')


raw files: 269
rows (metadata count): 60810792
available columns: ['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'status', 'borough', 'incident_zip', 'incident_address', 'street_name', 'latitude', 'longitude']
missing columns: none


In [5]:
def build_monthly_borough_summary(dataset, columns: list[str]) -> pd.DataFrame:
    """Stream the raw dataset and build a compact monthly summary."""
    if not columns:
        raise ValueError('No usable columns were found in the raw dataset.')

    parts: list[pd.DataFrame] = []
    scanner = dataset.scanner(columns=columns)
    for batch in scanner.to_batches():
        frame = batch.to_pandas()
        cleaned = clean_311_frame(frame)
        if cleaned.empty:
            continue

        cleaned['created_month'] = cleaned['created_date'].dt.to_period('M').astype(str)
        part = (
            cleaned.groupby(['created_month', 'borough', 'complaint_type'], dropna=False)
            .size()
            .reset_index(name='requests')
        )
        parts.append(part)

    if not parts:
        return pd.DataFrame(columns=['created_month', 'borough', 'complaint_type', 'requests'])

    summary = pd.concat(parts, ignore_index=True)
    summary = (
        summary.groupby(['created_month', 'borough', 'complaint_type'], dropna=False, as_index=False)['requests']
        .sum()
        .sort_values(['created_month', 'requests'], ascending=[True, False])
        .reset_index(drop=True)
    )
    return summary


monthly_summary = build_monthly_borough_summary(raw_311, available_columns)
summary_out = OUT_ROOT / '311_monthly_borough_complaint_summary.csv'
monthly_summary.to_csv(summary_out, index=False)

print(summary_out)
monthly_summary.head(10)


/home/map10194/Documents/urban-flood-stress/data/temporal/311/311_monthly_borough_complaint_summary.csv


,created_month,borough,complaint_type,requests
0,2010-01,Unspecified,HEATING,40152
1,2010-01,Unspecified,GENERAL CONSTRUCTION,12098
2,2010-01,Unspecified,PLUMBING,10701
3,2010-01,Unspecified,PAINT - PLASTER,7953
4,2010-01,Unspecified,NONCONST,5185
5,2010-01,Unspecified,ELECTRIC,3566
6,2010-01,BRONX,Noise - Residential,2997
7,2010-01,BROOKLYN,Noise - Residential,2794
8,2010-01,BROOKLYN,Street Condition,2673
9,2010-01,QUEENS,Street Condition,2298


In [6]:
top_complaints = (
    monthly_summary.groupby('complaint_type', dropna=False)['requests']
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

top_complaints

,complaint_type,requests
0,Noise - Residential,6497837
1,Illegal Parking,6048212
2,HEAT/HOT WATER,4403532
3,Blocked Driveway,2868516
4,Noise - Street/Sidewalk,2530678
5,UNSANITARY CONDITION,1707031
6,Request Large Bulky Item Collection,1704509
7,Street Condition,1575481
8,PLUMBING,1434838
9,Water System,1382998
